# 📈 03: Revenue Forecast Eval

Detailed performance tracking of the `XGBoost` + `Prophet` ensemble.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from components.B_revenue_forecast.features import build_forecast_features
from components.B_revenue_forecast.ensemble import ForecastEnsemble
from data.loader import load_transactions

df = load_transactions()
daily_df = build_forecast_features(df)

# Ensemble requires > 1 day, so we take our processed daily_df
# In the runner, we mocked data, here we visualize the series
print(f"Daily aggregated series length: {len(daily_df)}")

## 1. Daily Revenue Trend
Reviewing the ground truth target series.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(daily_df['ds'], daily_df['y'], marker='o', linestyle='-', color='teal')
plt.title("Daily Aggregated Revenue (amount_due_merchant)")
plt.xlabel("Date")
plt.ylabel("Revenue (NGN)")
plt.show()

## 2. Feature Correlations with Revenue
Which hourly characteristics correlate with higher revenue?

In [ ]:
corr = daily_df.drop(columns=['ds']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Feature Correlation Matrix")
plt.show()

## 3. Training & Performance
Since the current transaction snapshot might be small, we'll expand it with synthetic history if necessary to demonstrate the models.

In [ ]:
from components.B_revenue_forecast.xgboost_model import XGBoostForecastModel
from components.B_revenue_forecast.prophet_model import ProphetForecastModel

if len(daily_df) < 5:
    print("Expanding data for demonstration purposes...")
    base_row = daily_df.iloc[-1].to_dict()
    mocked_rows = []
    for i in range(30, 0, -1):
        row = base_row.copy()
        row['ds'] = row['ds'] - pd.Timedelta(days=i)
        row['y'] = row['y'] * np.random.uniform(0.7, 1.3)
        row['txn_count'] = int(row['txn_count'] * np.random.uniform(0.7, 1.3))
        mocked_rows.append(row)
    
    mock_df = pd.DataFrame(mocked_rows)
    daily_df = pd.concat([mock_df, daily_df]).reset_index(drop=True)
    # Simple refresh of some features
    daily_df['is_weekend'] = daily_df['ds'].dt.dayofweek.isin([5, 6]).astype(int)
    daily_df['y_lag_1'] = daily_df['y'].shift(1).fillna(0)

split_idx = int(len(daily_df) * 0.8)
train_df = daily_df.iloc[:split_idx]
test_df = daily_df.iloc[split_idx:]

xgb_model = XGBoostForecastModel()
prophet_model = ProphetForecastModel()
ensemble = ForecastEnsemble(xgb_model, prophet_model)

ensemble.fit(train_df)
print(f"Training complete. Model evaluated on {len(test_df)} days of history.")

## 4. Evaluation Metrics
Comparing the individual models vs the ensemble.

In [ ]:
xgb_eval = xgb_model.evaluate(test_df)
prop_eval = prophet_model.evaluate(test_df)
ens_preds = ensemble.predict(test_df)
actual = test_df['y'].values

from sklearn.metrics import mean_absolute_error
ens_mae = mean_absolute_error(actual, ens_preds)

performance = pd.DataFrame({
    'Model': ['XGBoost', 'Prophet', 'Ensemble'],
    'RMSE': [xgb_eval['rmse'], prop_eval['rmse'], np.sqrt(np.mean((actual-ens_preds)**2))],
    'MAPE': [xgb_eval['mape'], prop_eval['mape'], ens_mae / np.mean(actual) if np.mean(actual) != 0 else 1.0]
})
display(performance)

## 5. Visualizing the Forecast
Final predicted values for the lookback window.

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_df['ds'], actual, 'o-', label='Actual', color='black', alpha=0.5)
plt.plot(test_df['ds'], ens_preds, 's--', label='Ensemble Forecast', color='coral')
plt.title("Backtest Forecast vs Reality")
plt.legend()
plt.xticks(rotation=45)
plt.show()

## 6. Tomorrow's Narrative Output
The user-facing component of the forecast module.

In [ ]:
forecast_input = test_df.tail(1).copy()
forecast_input['ds'] = forecast_input['ds'] + pd.Timedelta(days=1)
pred_next = ensemble.predict(forecast_input)
print(ensemble.generate_narrative(forecast_input, pred_next))